# bayesflow_hpo Quickstart

A minimal end-to-end example that shows how to run **hyperparameter optimization** (HPO) for a [BayesFlow 2.x](https://bayesflow.org) amortized inference workflow.

We will:

1. Define a simple Gaussian **simulator** (prior + likelihood).
2. Build a BayesFlow **adapter** that maps raw simulation output to the format expected by the neural network.
3. Generate a fixed **validation dataset** for simulation-based calibration (SBC) diagnostics.
4. Launch an Optuna-backed **multi-objective optimization** run that searches over network architectures and training hyperparameters.

## 0. Setup

Install the package (editable mode from the repo root) and import the two libraries we need:
- **`bayesflow`** — the core amortized Bayesian inference framework (simulators, adapters, workflows),
- **`bayesflow_hpo`** — this package, which adds HPO search spaces, objectives, and validation utilities on top.

In [1]:
%pip install --quiet --upgrade -e ..

import bayesflow as bf
import bayesflow_hpo as hpo
import numpy as np

Note: you may need to restart the kernel to use updated packages.


INFO:bayesflow:Using backend 'torch'
When using torch backend, we need to disable autograd by default to avoid excessive memory usage. Use

with torch.enable_grad():
    ...

in contexts where you need gradients (e.g. custom training loops).
c:\Users\Matze\Documents\GitHub\bayesflow_projects\bayesflow_hpo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Simulator, Adapter & Validation Data

**Simulator** — We define a toy generative model with a 1-D Gaussian prior $\theta \sim \mathcal{N}(0, 1)$ and a Gaussian likelihood $x_i \mid \theta \sim \mathcal{N}(\theta, 1)$ producing 12 observations per dataset. This is deliberately simple so the notebook runs in seconds.

**Adapter** — The `bf.Adapter` tells BayesFlow how to reshape the raw simulation dictionaries into the tensor format the neural network expects:
- `.as_set(["x"])` marks observation vectors as exchangeable (order doesn't matter),
- `.rename("theta", "inference_variables")` maps the parameter to the inference target,
- `.concatenate(["x"], into="summary_variables")` stacks observations into the summary input.

**Validation data** — `hpo.generate_validation_dataset` draws a fixed batch of (parameter, data) pairs from the simulator. This dataset is reused across *every* HPO trial so that metric comparisons are fair (no noise from resampling).

In [2]:
def prior_fn():
    return {"theta": np.random.normal(0.0, 1.0, size=(1,)).astype("float32")}


def likelihood_fn(theta):
    theta_value = float(np.squeeze(theta))
    x = np.random.normal(theta_value, 1.0, size=(12, 1)).astype("float32")
    return {"x": x}


simulator = bf.simulators.make_simulator([prior_fn, likelihood_fn])
adapter = (
    bf.Adapter()
    .as_set(["x"])
    .rename("theta", "inference_variables")
    .concatenate(["x"], into="summary_variables", axis=-1)
)

validation_data = hpo.generate_validation_dataset(
    simulator=simulator,
    param_keys=["theta"],
    data_keys=["x"],
    sims_per_condition=100,
)

## 2. Run HPO

`hpo.optimize` is the main entry point. Under the hood it:

1. **Creates an Optuna study** with two objectives: *mean(calibration_error + nrmse)* (minimize) and *model size* (minimize) — a Pareto-style trade-off between accuracy and complexity.
2. **Samples hyperparameters** from a `CompositeSearchSpace` that covers the inference network (coupling flow), the summary network (DeepSet), and training settings (learning rate).
3. **Builds, trains, and validates** a fresh `bf.BasicWorkflow` for each trial, using SBC-based metrics on the fixed validation dataset.
4. **Reports results** back to Optuna, which guides future sampling via its TPE (Tree-structured Parzen Estimator) sampler.

Key arguments in this example:
| Argument | Value | Why |
|---|---|---|
| `n_trials` | 10 | Number of HPO configurations to try (increase for real use) |
| `epochs` | 30 | Enough gradient steps for meaningful metric signal |
| `batches_per_epoch` | 30 | 900 gradient steps per trial — fast demo |
| `max_param_count` | 500,000 | Reject very large models before training |
| `objective_metrics` | `["calibration_error", "nrmse"]` | Combines coverage quality with point-estimate accuracy |
| `objective_mode` | `"mean"` | Average both metrics into one scalar objective |
| `inference_conditions` | `["num_obs"]` | Tell the inference network to condition on sample size |

After optimization, `study.best_trials` returns the Pareto-optimal trial(s).

In [ ]:
import bayesflow_hpo as hpo

# Use a focused search space for the demo (coupling flow + deep set).
# The default also includes flow matching + set transformer, which are
# slower to converge on this toy problem.
search_space = hpo.CompositeSearchSpace(
    inference_space=hpo.CouplingFlowSpace(),
    summary_space=hpo.DeepSetSpace(),
    training_space=hpo.TrainingSpace(),
)

study = hpo.optimize(
    simulator=simulator,
    adapter=adapter,
    param_keys=["theta"],
    data_keys=["x", "num_obs"],
    validation_data=validation_data,
    inference_conditions=["num_obs"],
    search_space=search_space,
    n_trials=10,
    epochs=30,
    batches_per_epoch=30,
    max_param_count=500_000,
    objective_metrics=["calibration_error", "nrmse"],
    objective_mode="mean",
    show_progress_bar=False,
)

print(f"Trials: {len(study.trials)}")
print(f"Best values: {study.best_trials[0].values}")

## 3. Inspect Results

### 3.1 Pareto Front

The optimization minimizes *two* objectives simultaneously:
- **Objective 0** — calibration error (lower = better-calibrated posterior approximation),
- **Objective 1** — normalized parameter count (lower = smaller model).

These objectives trade off against each other: a very large network may achieve low calibration error but is expensive to run. The **Pareto front** shows the set of trials that are not dominated — no other trial is simultaneously smaller *and* better calibrated.

### 3.2 Hyperparameter Importance

After all trials have run, Optuna's **fANOVA-based importance estimator** assigns each hyperparameter a score reflecting how much its variation explains the variance in objective 0 (calibration error). High-importance hyperparameters are worth tuning carefully in follow-up studies.

### 3.3 Optimization History

The convergence plot shows how the best objective value evolves over successive trials. A healthy HPO run should trend downward — if it plateaus early, more trials won't help and you should widen the search space instead.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3.1 Pareto front: calibration error vs parameter count
hpo.plot_pareto_front(study, ax=axes[0])

# 3.2 Hyperparameter importance (fANOVA)
hpo.plot_param_importance(study, ax=axes[1])

# 3.3 Optimization history: best calibration error over trials
trained = [
    t for t in study.trials
    if t.values is not None and "rejected_reason" not in t.user_attrs
]
if trained:
    trained.sort(key=lambda t: t.number)
    trial_nums = [t.number for t in trained]
    cal_errors = [t.values[0] for t in trained]
    best_so_far = [min(cal_errors[: i + 1]) for i in range(len(cal_errors))]
    axes[2].scatter(trial_nums, cal_errors, alpha=0.4, label="Trial")
    axes[2].step(trial_nums, best_so_far, where="post", color="red", label="Best so far")
    axes[2].set_xlabel("Trial number")
    axes[2].set_ylabel("Calibration error")
    axes[2].set_title("Optimization history")
    axes[2].legend()

plt.tight_layout()
plt.show()

### 3.4 Tabular Summary

`trials_to_dataframe` converts completed trials into a `pandas.DataFrame` with one row per trial and columns for every hyperparameter, both objective values, and validation metrics (NRMSE, correlation, coverage, etc.). Budget-rejected trials are excluded by default.

The table is sorted by calibration error (the first objective) so the best architectures appear at the top.

In [ ]:
df = hpo.trials_to_dataframe(study)
print(f"Completed trials: {len(df)}")

# Show the most informative columns, sorted by calibration error
key_cols = ["trial_number", "calibration_error", "param_count", "nrmse", "correlation",
            "coverage_90", "coverage_95", "training_time_s"]
display_cols = [c for c in key_cols if c in df.columns]
df.sort_values("calibration_error")[display_cols].head(10)

### 3.5 Study Summary

`summarize_study` prints a concise overview: trial counts (trained, rejected, pruned, failed), the Pareto front, the best trial by calibration error, a top-k leaderboard, and the winning hyperparameters — everything you need to decide which configuration to use for downstream inference.

In [ ]:
hpo.summarize_study(study, top_k=5)